# Batch Layer - Gold: Analytics & Aggregations

**Objective:**
- Create business-ready analytics tables
- Multi-level aggregations
- Support BI tools and dashboards
- Enable ad-hoc analytics

**Key Databricks Features:**
✅ SQL analytics queries  
✅ Materialized aggregations  
✅ Performance partitioning  
✅ Clustering for optimization

## 1. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, countDistinct, min_by, max_by, collect_list

spark = SparkSession.builder.getOrCreate()

silver_table = "dev.silver.batch_item_properties"
gold_table_1 = "dev.gold.batch_item_property_summary"
gold_table_2 = "dev.gold.batch_property_statistics"

print("🎯 Creating Gold tables for analytics")

## 2. Gold Table 1: Item-Property Mapping Summary

In [ ]:
# Aggregation: Count properties per item
item_property_summary = spark.sql(f"""
SELECT 
  item_id,
  COUNT(DISTINCT property_id) as property_count,
  COUNT(*) as total_values,
  COLLECT_LIST(DISTINCT property_id) as property_ids,
  MIN(event_time) as first_seen,
  MAX(event_time) as last_seen,
  DATEDIFF(MAX(event_time), MIN(event_time)) as days_active,
  CURRENT_TIMESTAMP() as computed_at
FROM {silver_table}
GROUP BY item_id
ORDER BY property_count DESC
""")

# Write to Gold
item_property_summary.coalesce(1).write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(gold_table_1)
print(f"✅ {gold_table_1} created")
item_property_summary.display()

## 3. Gold Table 2: Property-Level Statistics

In [ ]:
# Property-centric analytics
property_stats = spark.sql(f"""
SELECT 
  property_id,
  COUNT(DISTINCT item_id) as items_with_property,
  COUNT(*) as total_property_records,
  COUNT(DISTINCT property_value) as distinct_values,
  MIN(event_time) as first_seen,
  MAX(event_time) as last_seen,
  ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 4) as frequency_ratio,
  CURRENT_TIMESTAMP() as computed_at
FROM {silver_table}
GROUP BY property_id
ORDER BY total_property_records DESC
""")

property_stats.coalesce(1).write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(gold_table_2)
print(f"✅ {gold_table_2} created")
property_stats.display()

## 4. Analytics Query Examples

In [ ]:
# Top 10 most enriched items
spark.sql(f"""
SELECT * FROM {gold_table_1}
ORDER BY property_count DESC
LIMIT 10
""").display()

In [ ]:
# Most common properties
spark.sql(f"""
SELECT * FROM {gold_table_2}
ORDER BY total_property_records DESC
LIMIT 10
""").display()

## 5. Data Quality & Lineage

In [ ]:
# Summary stats
spark.sql(f"""
SELECT 
  '{gold_table_1}' as table_name,
  COUNT(*) as record_count,
  COUNT(DISTINCT item_id) as unique_items,
  MAX(computed_at) as last_refresh
FROM {gold_table_1}
UNION ALL
SELECT 
  '{gold_table_2}' as table_name,
  COUNT(*) as record_count,
  COUNT(DISTINCT property_id) as unique_properties,
  MAX(computed_at) as last_refresh
FROM {gold_table_2}
""").display()